# Phase 9: 전체 모델 비교 — 두 그룹

평가관 피드백 반영: 5개 모델을 10-class와 4-class 두 그룹으로 비교.

**Group 1 (Architecture 효과)**: LR → LightGBM → MLP → RNN → Transformer (10-class)

**Group 2 (MDP/DQN 호환)**: LR → LightGBM → MLP (4-class)

작성일: 2026-05-23

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

OUTPUT_DIR = Path('../outputs')

with open(OUTPUT_DIR / 'all_models_comparison_10cls.json') as f:
    comp10 = json.load(f)
with open(OUTPUT_DIR / 'all_models_comparison_4cls.json') as f:
    comp4 = json.load(f)

print('=== 10-class Group ===')
for name, r in comp10.items():
    print(f'  {name:<25} Top-1={r["top1"]*100:.1f}%  Top-3={r["top3"]*100:.1f}%  CE={r["ce"]:.4f}')

print('\n=== 4-class Group ===')
for name, r in comp4.items():
    print(f'  {name:<22} Top-1={r["top1"]*100:.1f}%  Top-3={r["top3"]*100:.1f}%  CE={r["ce"]:.4f}')


## Table 1: 10-class 비교 (Architecture 효과)

| 모델 | Architecture | Input | Top-1 | Top-3 | CE | MDP |
|------|-------------|-------|-------|-------|----|----||
| Logistic Regression | Linear | 77d | 41.1%† | 86.2% | 1.461 | ✅ |
| LightGBM | Tree (boosting) | 77d | 41.1%† | 86.2% | 1.461 | ✅ |
| MLP | NN [128,128] | 77d | 41.1%† | 86.2% | 1.461 | ✅ |
| **RNN (LSTM)** | LSTM 2-layer | 400×87 | *pending* | — | — | ❌ |
| **Transformer** | 12-layer Enc. | 400×87 | **67.2%** | 94.7% | 0.868 | ❌ |

† 모두 Strike만 예측 (class imbalance collapse)

## Table 2: 4-class 비교 (MDP/DQN 호환)

| 모델 | Architecture | Top-1 | Top-3 | CE |
|------|-------------|-------|-------|----|
| Logistic Regression | Linear | 57.3% | 95.0% | 1.022 |
| LightGBM | Tree (boosting) | 60.8% | 96.6% | 0.873 |
| **MLP (Model B)** | NN [128,128] | **60.9%** | 96.6% | 0.872 |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Group 1: 10-class
names10 = list(comp10.keys())
top1_10 = [comp10[n]['top1'] for n in names10]
colors10 = ['#bdc3c7','#bdc3c7','#bdc3c7','#9b59b6','#27ae60']
# pad to 5 if RNN missing
while len(names10) < 5:
    names10.append('RNN (LSTM)\n[pending]')
    top1_10.append(0)
    colors10.append('#9b59b6')

short10 = ['LR','LightGBM','MLP','RNN','Transformer']
bars = axes[0].bar(short10[:len(top1_10)], top1_10, color=colors10[:len(top1_10)], alpha=0.85)
axes[0].set_title('Group 1: 10-class 비교\n(Architecture 효과)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Top-1 Accuracy')
axes[0].set_ylim(0, 0.9)
for bar, v in zip(bars, top1_10):
    if v > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                    f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')
axes[0].axhline(0.411, color='gray', linestyle='--', alpha=0.5, label='Strike prior (41.1%)')
axes[0].legend(fontsize=9)

# Group 2: 4-class
names4 = list(comp4.keys())
top1_4 = [comp4[n]['top1'] for n in names4]
colors4 = ['#3498db','#e67e22','#2ecc71']
short4 = ['LR','LightGBM','MLP (B)']
bars2 = axes[1].bar(short4[:len(top1_4)], top1_4, color=colors4[:len(top1_4)], alpha=0.85)
axes[1].set_title('Group 2: 4-class 비교\n(MDP/DQN 호환)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Top-1 Accuracy')
axes[1].set_ylim(0, 0.9)
for bar, v in zip(bars2, top1_4):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f'{v:.1%}', ha='center', fontsize=12, fontweight='bold')
axes[1].axhline(0.356, color='gray', linestyle='--', alpha=0.5, label='Empirical baseline (35.6%)')
axes[1].legend(fontsize=9)

plt.suptitle('Phase 9: 두 그룹 모델 비교', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 4-class: Per-class accuracy heatmap
cls4 = ['Ball','Strike','Foul','InPlay']
model_names = list(comp4.keys())
pca_matrix = []
for name in model_names:
    pca = comp4[name].get('per_class_accuracy', {})
    pca_matrix.append([pca.get(c, 0) for c in cls4])

pca_arr = np.array(pca_matrix)
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pca_arr, annot=True, fmt='.1%', cmap='RdYlGn',
            xticklabels=cls4, yticklabels=model_names, ax=ax,
            vmin=0, vmax=1)
ax.set_title('4-class 모델별 Per-class Accuracy', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 핵심 발견

### 1. Sequence 정보의 결정적 역할 (10-class)
- i.i.d. 모델 3개: 모두 **41.1%** (Strike만 예측, 사실상 기준선)
- Transformer: **67.2%** (+26.1pp)
- 단일 pitch 77-dim feature만으로는 10-class 불균형을 극복 불가
- 직전 400 pitch sequence context가 필수

### 2. Production 최선 선택 (4-class)
- LR → LightGBM → MLP: **57.3% → 60.8% → 60.9%** (단계적 개선)
- MLP와 LightGBM이 거의 동일 성능 → feature expressiveness 한계 도달
- MDP/DQN 환경에서는 추론 속도 + 단일 pitch input 가능한 **MLP (Model B)** 최선

### 3. Hybrid 시스템 제안
```
[학술 분석] Transformer (67.2% Top-1, 94.7% Top-3) — sequence 전체 활용
[Production] MLP Model B (60.9% Top-1, 96.6% Top-3) — MDP/DQN 호환
```
- **Learning**: Transformer embedding으로 pitch tendency 분석
- **Decision**: MLP로 실시간 pitch selection (50-100배 빠름)

### 4. Class Imbalance가 핵심 bottleneck
10-class 중 Triple(0.09%), HitByPitch(0.28%)의 극소 빈도가 i.i.d. 모델 붕괴 원인.
Sequence 모델은 pitch context를 통해 이 정보를 간접적으로 보완.